In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader

hyperbook_docs = CSVLoader(
    file_path='../../data/raw/hyperbook.csv', 
    content_columns=['content'], 
    metadata_columns=['url', 'title', 'authors'],
    encoding='utf-8'
).load()

In [4]:
from pathlib import Path
from langchain_community.document_loaders.pdf import PyPDFLoader

textbooks_docs = []

ball_metadata = {
    'url': 'https://mcgill.on.worldcat.org/oclc/180470082',
    'title': "Shapes: Nature's patterns: a tapestry in three parts",
    'authors': 'Philip Ball'
}

def set_metadata(doc):
    doc.metadata = ball_metadata
    return doc

# Temporary — Only experimenting with three chapters from Shapes by Ball, P.
for chapter_path in Path('../../data/raw/textbooks/shapes_ball').iterdir():
    chapter_docs = PyPDFLoader(chapter_path).load()
    chapter_docs = list(map(set_metadata, chapter_docs))
    textbooks_docs = textbooks_docs + chapter_docs 

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = hyperbook_docs + textbooks_docs

# Chunk parameters
chunk_size = 256
chunk_overlap = 32

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
doc_splits = text_splitter.split_documents(docs)

In [6]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")
vector_store = Chroma(collection_name='bien200', embedding_function=embeddings, persist_directory='../../embeddings/chroma_db')

In [14]:
from uuid import uuid4

def split_list(input_list, chunk_size=5461):
    for i in range(0, len(input_list), chunk_size):
        yield input_list[i:i + chunk_size]

uuids = [str(uuid4()) for _ in range(len(doc_splits))]

len(list(split_list(list(zip(doc_splits, uuids))))[0][0])

2

In [15]:
from uuid import uuid4

def split_list(input_list, chunk_size=5461):
    for i in range(0, len(input_list), chunk_size):
        yield input_list[i:i + chunk_size]

for batch_docs in split_list(doc_splits):
    batch_ids = [str(uuid4()) for _ in range(len(doc_splits))]
    vector_store.add_documents(documents=batch_docs, ids=batch_ids)

In [16]:
dense_retriever = vector_store.as_retriever()

In [17]:
dense_retriever.invoke('signs for recognizing extraterrestrial life')

[Document(id='12396067-9f31-4c09-aa7b-88194002fe69', metadata={'title': "Shapes: Nature's patterns: a tapestry in three parts", 'authors': 'Philip Ball', 'url': 'https://mcgill.on.worldcat.org/oclc/180470082'}, page_content='The Shapes of Things\nPattern and Form\nARRIVING on Earth, the aliens approach the first thing they see and utter the familiar words: ‘take \nme to your leader’ ( ). Like many jokes, this one offers a damning critique. It undermines the Fig. 1.1\nvenerable and serious scientific quest to find life on other worlds, exploding the question of ‘how \nwould we  if we found it?’ by answering that we tend to imagine it will look like us.know\nNow, let me assure you that astrobiologists (as scientists who study aliens are called nowadays) \nare not really that foolish. They do not imagine for a moment that when we touch down on another \ninhabited world, we will be greeted by envoys who look like Leonard Nimoy. Indeed, if there is life \nin those parts of our own solar sys